# 02 — Data Quality and Temporal Split

## 1. Objective

Assess the quality, consistency, and temporal integrity of the SWaT dataset before exploratory analysis and anomaly-detection modelling.

The assessment covers data provenance, schema consistency, missing and invalid values, duplicate records, timestamp ordering, label integrity, and reconciliation between the separate and merged CSV files.

The findings will guide data preparation and the design of chronological evaluation partitions. Any data-quality issue that could affect model development or evaluation will be documented before proceeding.

## 2. Scope

This notebook will:

- Establish the dataset files and their structural characteristics.
- Verify column names, schema consistency, and data types.
- Assess missing values, invalid values, and duplicate records.
- Investigate timestamp ordering and temporal continuity.
- Examine the distribution and consistency of anomaly labels.
- Reconcile `normal.csv`, `attack.csv`, and `merged.csv`.
- Define an appropriate chronological partitioning strategy.
- Document data-quality limitations and their implications for subsequent analysis.

## 3. Data Handling Principles

- Preserve the original files in `data/raw/` without modification.
- Use the existing project paths rather than hardcoded machine-specific paths.
- Process large CSV files in chunks when a complete in-memory load is unnecessary.
- Distinguish verified findings from unresolved questions.
- Preserve temporal order when designing evaluation partitions.
- Document any preprocessing decisions that could affect anomaly detection or evaluation.

## 4. Expected Outcome

By the end of this notebook, the dataset's principal quality issues and temporal characteristics should be documented, file consistency should be assessed, and the rationale for the evaluation partitions should be established.

The notebook will provide the data-quality evidence needed before proceeding to **Notebook 03 — Exploratory Analysis**.

In [10]:
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        directory
        for directory in (CURRENT_DIR, *CURRENT_DIR.parents)
        if (directory / "data" / "raw").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate data/raw from the current directory: {CURRENT_DIR}"
    )

DATA_DIR = PROJECT_ROOT / "data" / "raw"

FILES = {
    "normal": DATA_DIR / "normal.csv",
    "attack": DATA_DIR / "attack.csv",
    "merged": DATA_DIR / "merged.csv",
}

missing_files = [
    path for path in FILES.values()
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Required dataset files not found:\n"
        + "\n".join(str(path) for path in missing_files)
    )

# print(f"Project root: {PROJECT_ROOT}")
# print(f"Raw data directory: {DATA_DIR}")

for name, path in FILES.items():
    print(f"{name}: {path.name} ({path.stat().st_size:,} bytes)")

normal: normal.csv (402,374,633 bytes)
attack: attack.csv (15,162,438 bytes)
merged: merged.csv (426,855,390 bytes)


## 5. Dataset Inventory

The raw dataset directory contains three CSV files: `normal.csv`, `attack.csv`, and `merged.csv`.

| File | Size (bytes) | Intended role |
|---|---:|---|
| `normal.csv` | 402,374,633 | Records designated as normal operation |
| `attack.csv` | 15,162,438 | Records associated with attack scenarios |
| `merged.csv` | 426,855,390 | Combined dataset containing normal and attack records |

The files are stored in `data/raw/`. Their existence and file sizes have been verified programmatically.

These checks establish the initial file inventory only. They do not establish that the files contain valid records, that `merged.csv` is an exact combination of the other two files, or that their timestamps are correctly ordered. Those properties require separate validation.

In [11]:
import csv

schemas = {}

for name, path in FILES.items():
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        reader = csv.reader(file)
        headers = next(reader)

    normalized_headers = [header.strip() for header in headers]

    schemas[name] = normalized_headers

    print(f"{name}: {len(headers)} columns")

    if len(normalized_headers) != len(set(normalized_headers)):
        raise ValueError(f"{name} contains duplicate column names after trimming whitespace.")

reference_schema = schemas["normal"]

for name, schema in schemas.items():
    if schema != reference_schema:
        raise ValueError(f"{name} has a different column order or schema from normal.csv.")

print("\nPASS: All three files have matching column names and column order.")
print("PASS: No duplicate column names were found after trimming whitespace.")

normal: 53 columns
attack: 53 columns
merged: 53 columns

PASS: All three files have matching column names and column order.
PASS: No duplicate column names were found after trimming whitespace.


## 6. Schema Consistency

All three raw CSV files contain **53 columns**. After removing leading and trailing whitespace from the column names, the files have identical column names and column order.

No duplicate column names were identified after normalization.

The raw headers contain leading whitespace, so column names should be normalized during analysis rather than modifying the original CSV files. The raw files remain preserved unchanged.

Schema consistency is therefore **PASS** across `normal.csv`, `attack.csv`, and `merged.csv`.

In [12]:
import pandas as pd

CHUNK_SIZE = 100_000

missing_results = {}

for name, path in FILES.items():
    missing_counts = pd.Series(dtype="int64")
    row_count = 0

    for chunk in pd.read_csv(path, chunksize=CHUNK_SIZE):
        chunk.columns = chunk.columns.str.strip()
        row_count += len(chunk)

        chunk_missing = chunk.isna().sum()

        if missing_counts.empty:
            missing_counts = chunk_missing.astype("int64")
        else:
            missing_counts = missing_counts.add(chunk_missing, fill_value=0).astype("int64")

    missing_results[name] = {
        "rows": row_count,
        "missing_total": int(missing_counts.sum()),
        "columns_with_missing": missing_counts[missing_counts > 0].sort_values(ascending=False),
    }

    print(f"{name}:")
    print(f"  rows: {row_count:,}")
    print(f"  total missing values: {int(missing_counts.sum()):,}")
    print(f"  columns with missing values: {len(missing_counts[missing_counts > 0])}")

    if (missing_counts > 0).any():
        print(missing_counts[missing_counts > 0])
    else:
        print("  No missing values found.")

    print()

normal:
  rows: 1,387,098
  total missing values: 6,942,600
  columns with missing values: 7
MV101     991800
AIT201    991800
MV201     991800
P201      991800
P202      991800
P204      991800
MV303     991800
dtype: int64

attack:
  rows: 54,621
  total missing values: 0
  columns with missing values: 0
  No missing values found.

merged:
  rows: 1,441,719
  total missing values: 6,942,600
  columns with missing values: 7
MV101     991800
AIT201    991800
MV201     991800
P201      991800
P202      991800
P204      991800
MV303     991800
dtype: int64



## 7. Missing Value Assessment

The missing-value assessment identified missing observations in seven variables:

- `MV101`
- `AIT201`
- `MV201`
- `P201`
- `P202`
- `P204`
- `MV303`

Each affected variable contains **991,800 missing values** in `normal.csv`, giving **6,942,600 missing cells** across the seven variables.

`attack.csv` contains **no missing values**. The same 6,942,600 missing cells are present in `merged.csv`, which is consistent with the missing values originating from the normal-operation records.

These missing values are not removed or imputed at this stage. Their temporal and process context must be investigated before determining whether they represent genuine sensor/actuator availability, structural missingness, or a data-quality problem.

**Status: Investigation required.**

In [ ]:
MISSING_COLUMNS = [
    "MV101", "AIT201", "MV201", "P201",
    "P202", "P204", "MV303",
]

with FILES["normal"].open("r", encoding="utf-8-sig", newline="") as file:
    raw_headers = next(csv.reader(file))

selected_columns = [
    header for header in raw_headers
    if header.strip() == "Timestamp"
    or header.strip() in MISSING_COLUMNS
]

if len(selected_columns) != len(MISSING_COLUMNS) + 1:
    raise ValueError(
        f"Expected {len(MISSING_COLUMNS) + 1} columns, "
        f"but found {len(selected_columns)}."
    )

missing_time_results = []
total_rows_with_missing_values = 0
total_invalid_timestamps = 0

for chunk in pd.read_csv(
    FILES["normal"],
    usecols=selected_columns,
    chunksize=CHUNK_SIZE,
    encoding="utf-8-sig",
):
    chunk.columns = chunk.columns.str.strip()

    chunk["Timestamp"] = pd.to_datetime(
        chunk["Timestamp"].str.strip(),
        format="%d/%m/%Y %I:%M:%S %p",
        errors="coerce",
    )

    total_invalid_timestamps += int(chunk["Timestamp"].isna().sum())

    missing_mask = chunk[MISSING_COLUMNS].isna().any(axis=1)
    affected = chunk.loc[missing_mask, "Timestamp"]

    total_rows_with_missing_values += int(missing_mask.sum())

    if missing_mask.any():
        missing_time_results.append({
            "first_timestamp": affected.min(),
            "last_timestamp": affected.max(),
            "affected_rows": int(missing_mask.sum()),
        })

print(f"Total rows with missing values: {total_rows_with_missing_values:,}")
print(f"Total invalid timestamps: {total_invalid_timestamps:,}")
print(f"Chunks containing missing values: {len(missing_time_results)}")

if missing_time_results:
    print(
        "Overall first timestamp:",
        min(r["first_timestamp"] for r in missing_time_results),
    )
    print(
        "Overall last timestamp:",
        max(r["last_timestamp"] for r in missing_time_results),
    )

if total_rows_with_missing_values != 991_800:
    raise ValueError(
        "The observed number of rows with missing values does not match "
        "the previously recorded count of 991,800."
    )

if total_invalid_timestamps:
    raise ValueError(
        f"Found {total_invalid_timestamps:,} invalid timestamps; "
        "investigate before interpreting the time range."
    )

Chunks containing missing values: 11
[{'first_timestamp': Timestamp('2015-12-22 16:00:00'), 'last_timestamp': Timestamp('2015-12-22 17:18:21'), 'affected_rows': 4702}, {'first_timestamp': Timestamp('2015-12-22 17:18:22'), 'last_timestamp': Timestamp('2015-12-23 21:05:01'), 'affected_rows': 100000}, {'first_timestamp': Timestamp('2015-12-23 21:05:02'), 'last_timestamp': Timestamp('2015-12-25 00:51:41'), 'affected_rows': 100000}, {'first_timestamp': Timestamp('2015-12-25 00:51:42'), 'last_timestamp': Timestamp('2015-12-26 04:38:21'), 'affected_rows': 100000}, {'first_timestamp': Timestamp('2015-12-26 04:38:22'), 'last_timestamp': Timestamp('2015-12-27 08:25:01'), 'affected_rows': 100000}, {'first_timestamp': Timestamp('2015-12-22 16:30:00'), 'last_timestamp': Timestamp('2015-12-28 09:59:59'), 'affected_rows': 100000}, {'first_timestamp': Timestamp('2015-12-22 18:41:42'), 'last_timestamp': Timestamp('2015-12-23 22:28:21'), 'affected_rows': 100000}, {'first_timestamp': Timestamp('2015-12-2